# 02b · Train KAN-SAE on the same GraphCast data

**Run this after notebook 02 has saved the TopK run. Choose a GPU (T4 or larger), connect the same Drive account, and select Runtime → Run all.** No GraphCast run is needed here.

The trainer reads the exact training/validation files and checksums from your TopK run. It uses 4,096 signed spline features, with an L1 penalty instead of TopK selection. The default is 20 epochs; the paper used 100 epochs on a different model and dataset. These are starting settings for our course experiment.

Results go into a separate `kan_runs/kan_baseline` folder. Rerunning resumes the last completed epoch and restores the calibrated knots. An interrupted partial epoch is repeated.

## Step 1 — Connect Drive and load the shared project code

In [ ]:
# @title Connect Drive and load the project
import json
from google.colab import drive

import os
import subprocess
import sys
from pathlib import Path

# One immutable code version is shared by all six notebooks.
RELEASE = 'b00fd40e75059f2ac345c70b273718a1cd945b94'
REPO = Path('/content') / ('ams691_project_' + RELEASE[:12])
if not (REPO / '.git').is_dir():
    subprocess.run(['git', 'init', '-q', str(REPO)], check=True)
head = subprocess.run(['git', '-C', str(REPO), 'rev-parse', '--verify', 'HEAD'],
                      capture_output=True, text=True)
if head.stdout.strip() != RELEASE:
    subprocess.run(['git', '-C', str(REPO), 'fetch', '-q', '--depth', '1',
                    'https://github.com/observerhan/graphcast-interpretability.git', RELEASE], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', '-q', '--detach', RELEASE], check=True)
sys.path.insert(0, str(REPO / 'scripts'))
sys.path.insert(0, str(REPO / 'src'))

drive.mount('/content/drive')
DRIVE_PROJECT = Path('/content/drive/MyDrive/AMS691_GraphCast')

## Step 2 — Training settings

Keep the defaults for the first run. `REFERENCE_TOPK_RUN` names the TopK run whose split we reuse. To continue training later, increase `EPOCHS` and rerun all cells. Keep `L1_RAMP_EPOCHS` and the other settings unchanged; changing them requires a new `RUN_NAME`.

Lambda increases from `L1_START` to `L1_END` over the fixed ramp, then stays at `L1_END`. Extending `EPOCHS` does not restart or stretch the ramp.

In [ ]:
# @title KAN-SAE settings
EPOCHS = 20  # @param {type:"integer"}
RUN_NAME = 'kan_baseline'  # @param {type:"string"}
SAE_EXPERIMENT = 'monthly_2017_2018'  # @param {type:"string"}
REFERENCE_TOPK_RUN = 'topk_baseline'  # @param {type:"string"}
N_FEATURES = 4096
BATCH_SIZE = 256
LEARNING_RATE = 1e-4
SEED = 0
CALIBRATION_SAMPLES = 50000
L1_START = 5e-5
L1_END = 1e-4
L1_RAMP_EPOCHS = 20
ACTIVITY_THRESHOLD = 1e-6

from train_kan_sae import TrainingConfig, prepare_data, KANTrainer
import torch
for name in [RUN_NAME, SAE_EXPERIMENT, REFERENCE_TOPK_RUN]:
    if not name or Path(name).name != name or name in {'.', '..'}:
        raise ValueError('Use simple folder names for the experiment and runs.')
config = TrainingConfig(latent=N_FEATURES, batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE, seed=SEED, calibration_samples=CALIBRATION_SAMPLES,
    l1_start=L1_START, l1_end=L1_END, l1_ramp_epochs=L1_RAMP_EPOCHS,
    activity_threshold=ACTIVITY_THRESHOLD)
config.validate()
PROJECT = DRIVE_PROJECT / SAE_EXPERIMENT
RUN_DIR = PROJECT / 'kan_runs' / RUN_NAME
REFERENCE_SPLIT = PROJECT / 'topk_runs' / REFERENCE_TOPK_RUN / 'data_split.json'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type != 'cuda':
    raise RuntimeError('Select a GPU runtime, then rerun all cells.')
print('Training GPU:', torch.cuda.get_device_name())
print('Results folder:', RUN_DIR)

## Step 3 — Verify and cache the same data used for TopK

The shared script checks every file against the TopK run's SHA-256 checksum. Calibration and the initial mean use training dates only; the separate validation dates select the best reconstruction checkpoint. Files are copied to Colab's temporary disk for faster training.

In [ ]:
# @title Load the reference split and cache activations
data = prepare_data(PROJECT / 'activations_layer8', REFERENCE_SPLIT,
                    Path('/content/ams691_kan_data') / SAE_EXPERIMENT)
if data.split['source']['shape'] != [40962, 512]:
    raise ValueError('This notebook expects the full GraphCast layer-8 activations.')
print('Training dates:', data.split['training_dates'])
print('Validation dates:', data.split['validation_dates'])
print('Training vectors:', len(data.train_files) * 40962)
print('Validation vectors:', len(data.val_files) * 40962)

## Step 4 — Initialize once, or restore the previous run

For a new run, the trainer computes the training mean and selects up to 50,000 training rows without replacement to calibrate the spline grids. It saves the exact row indices. For a resumed run, it restores the model, optimizer, knots, history, and best checkpoint without recalibrating.

The implementation is in `scripts/train_kan_sae.py`; the model is in `src/graphcast_interpretability/kan_sae.py`.

In [ ]:
# @title Initialize or resume KAN-SAE
trainer = KANTrainer(data, RUN_DIR, config, device=device, code_revision=RELEASE)
print('Last completed epoch:', trainer.epoch)
print('Calibration rows:', trainer.calibration['count'])
print('Initial validation error:', trainer.history[0]['validation_error'])

## Step 5 — Train and save to Drive

**Training happens in this cell.** Each batch updates the encoder, decoder, biases, and spline coefficients. Decoder columns are normalized after every optimizer step. At every completed epoch, `last.pt` saves the resumable state and `best.pt` records the lowest validation reconstruction error.

The reported activity count uses `abs(code) > ACTIVITY_THRESHOLD`, so negative features count as active. L1 encourages small codes but does not enforce exactly 32 nonzero features.

In [ ]:
# @title Train KAN-SAE
history = trainer.fit(EPOCHS)
print('Completed epochs:', trainer.epoch)
print('Best validation epoch:', trainer.best_epoch)

## Step 6 — Inspect the learning curve and selected checkpoint

The reconstruction curve uses the same per-vector squared error as the TopK notebook. The second plot shows the sum of absolute feature values. `validation_clamped_fraction` reports how often encoder values fall outside the calibrated spline range; a large fraction is a reason to inspect the learned model.

The checkpoint is selected by validation reconstruction error, not cyclone test F1.

In [ ]:
# @title Show results and load the best KAN-SAE
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from sae_checkpoint import load_model
summary = trainer.summarize()
trainer.plot_history()
plt.show()
display(pd.DataFrame([summary]))
best = torch.load(RUN_DIR / 'best.pt', map_location='cpu', weights_only=True)
model = load_model(best, device).eval()
print('Best model:', RUN_DIR / 'best.pt')
print('History:', RUN_DIR / 'history.csv')
if summary['best_epoch'] == 0:
    print('Training has not improved on the initial mean reconstruction yet. Inspect settings before interpreting features.')

Next, open [notebook 04](https://colab.research.google.com/github/observerhan/graphcast-interpretability/blob/kan-sae-training/04_probe_topk_cyclones.ipynb) to fit and evaluate the cyclone probes. Its default `SAE_TYPE='kan_sae'` selects this model. Keep the same experiment and run name. Use `SAE_TYPE='topk'` to evaluate the baseline instead.

If notebook 03's labeled activations already exist, reuse them. Notebook 05 also supports KAN features and displays their signed values with a color scale centered on zero.